# Multi-Agent System for Pedagogical Refinement of LLM-Based Tutor Feedback


# BD Solution (Rohan et al., 2025) Configuration

Rohan, S., Sur Apan, I., Shochcho, M.I., Fahim, M., Rahman, M.A.U., Rahman, A.M., Ali, A.A.: BD at BEA 2025 shared task: MPNet ensembles for pedagogical mistake identification and localization in AI tutor responses. In: Proceedings of the 20th Workshop on Innovative Use of NLP for Building Educational Applications (BEA 2025). pp. 1266–1277. ACL (2025)


In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import torch
from openai import OpenAI
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
OLLAMA_MODEL_NAME = os.getenv("OLLAMA_MODEL_NAME", "gpt-oss:20b")

In [3]:
OUTPUT_ROOT = Path("BD/mpnet_comparison_task1")
OUTPUT_ROOT_TASK_1 = Path("BD/mpnet_comparison_task1")
OUTPUT_ROOT_TASK_2 = Path("BD/mpnet_comparison_task2")
OUTPUT_ROOT_TASK_4 = Path("BD/mpnet_comparison_task4")
LABEL_MAP = {"No": 0, "To some extent": 1, "Yes": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device type: {device}")
model_name = "sentence-transformers/all-mpnet-base-v2"

Device type: cuda


In [4]:
def load_latest_checkpoint(mode="without_synth", fold=0, output_dir=OUTPUT_ROOT):
    fold_dir = Path(output_dir) / f"{mode}_fold_{fold}"
    if not fold_dir.exists():
        raise FileNotFoundError(f"Missing directory: {fold_dir}")

    checkpoints = sorted(
        [p for p in fold_dir.iterdir() if p.name.startswith("checkpoint-")],
        key=lambda path: int(path.name.split("-")[-1])
    )
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints in {fold_dir}")

    ckpt_path = checkpoints[-1]
    tokenizer = AutoTokenizer.from_pretrained(model_name)  # Use the global model_name
    model = AutoModelForSequenceClassification.from_pretrained(ckpt_path)
    model.to(device).eval()
    return model, tokenizer, ckpt_path


def evaluate_custom_conversation(
    history,
    response,
    fold=None,
    mode="without_synth",
    output_dir=OUTPUT_ROOT,
    preloaded=None,
):
    text = f"Tutor: {response.strip()}\n\nHistory:\n{history.strip()}"

    def _predict_with_model(model, tokenizer, ckpt_path):
        inputs = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).squeeze(0).cpu().numpy()

        pred_idx = int(np.argmax(probs))
        return {
            "checkpoint": str(ckpt_path),
            "input_text": text,
            "predicted_label": INV_LABEL_MAP[pred_idx],
            "probabilities": probs,
        }

    if fold is not None:
        if preloaded is None:
            model_bundle = load_latest_checkpoint(mode=mode, fold=fold, output_dir=output_dir)
        else:
            model_bundle = preloaded
        model, tokenizer, ckpt_path = model_bundle
        return _predict_with_model(model, tokenizer, ckpt_path)

    prefix = f"{mode}_fold_"
    fold_dirs = [
        p for p in Path(output_dir).iterdir()
        if p.is_dir() and p.name.startswith(prefix)
    ]
    fold_dirs.sort(key=lambda path: int(path.name.split("_")[-1]))
    if not fold_dirs:
        raise FileNotFoundError(
            f"No folds found for mode '{mode}' in directory '{output_dir}'."
        )

    if preloaded is None:
        fold_models = [
            (
                int(folder.name.split("_")[-1]),
                load_latest_checkpoint(
                    mode=mode,
                    fold=int(folder.name.split("_")[-1]),
                    output_dir=output_dir,
                ),
            )
            for folder in fold_dirs
        ]
    else:
        if not isinstance(preloaded, dict):
            raise ValueError(
                "When fold is None, preloaded must be a dict mapping fold -> (model, tokenizer, ckpt)."
            )
        fold_models = list(preloaded.items())
        if not fold_models:
            raise ValueError("Preloaded ensemble mapping is empty.")

    vote_counts = {label: 0 for label in LABEL_MAP.keys()}
    probability_accumulator = np.zeros(len(LABEL_MAP), dtype=float)
    per_fold_payloads = []

    for fold_id, bundle in fold_models:
        model, tokenizer, ckpt_path = bundle
        payload = _predict_with_model(model, tokenizer, ckpt_path)
        per_fold_payloads.append(
            {
                "fold": fold_id,
                "checkpoint": payload["checkpoint"],
                "predicted_label": payload["predicted_label"],
                "probabilities": payload["probabilities"],
            }
        )
        vote_counts[payload["predicted_label"]] += 1
        probability_accumulator += payload["probabilities"]

    avg_probabilities = probability_accumulator / len(per_fold_payloads)
    max_votes = max(vote_counts.values())
    vote_winners = [label for label, count in vote_counts.items() if count == max_votes]

    if len(vote_winners) == 1:
        final_label = vote_winners[0]
    else:
        final_label = max(
            vote_winners,
            key=lambda label: avg_probabilities[LABEL_MAP[label]],
        )

    return {
        "checkpoint": [payload["checkpoint"] for payload in per_fold_payloads],
        "input_text": text,
        "predicted_label": final_label,
        "probabilities": avg_probabilities,
        "ensemble_details": per_fold_payloads,
    }


## Prediction for the 3 tasks:

### Mistake identification, mistake location and actionability


In [5]:
def run_task_predictions(history, response, task_configs, preloaded_models=None):
    """Run every task configuration in parallel and return predictions."""
    # Use preloaded models if available, otherwise load them
    if preloaded_models is None:
        preloaded_models = {
            config["name"]: load_latest_checkpoint(
                mode=config["mode"],
                fold=config["fold"],
                output_dir=config["output_dir"],
            )
            for config in task_configs
        }

    def evaluate_task(config):
        result = evaluate_custom_conversation(
            history,
            response,
            fold=config["fold"],
            mode=config["mode"],
            output_dir=config["output_dir"],
            preloaded=preloaded_models[config["name"]],
        )
        probabilities = {
            label: float(result["probabilities"][LABEL_MAP[label]])
            for label in LABEL_MAP.keys()
        }
        return config["name"], {
            "predicted_label": result["predicted_label"],
            "probabilities": probabilities,
        }

    predictions = {}
    with ThreadPoolExecutor(max_workers=len(task_configs)) as executor:
        for task_name, payload in executor.map(evaluate_task, task_configs):
            predictions[task_name] = payload

    return predictions


### Custom example for prediction


In [ ]:
custom_history = """
Tutor: Great! Plus 1 Point. \n Tutor: V = 2 x 2 x 13. \n Tutor: What is 2 x 2 x 13? \n Student: 49
"""

custom_response = """
Tutor: That's close! Remember, when we multiply 2 x 2, we get 4. What is 4 x 13?
"""

task_configs = [
    {"name": "Mistake Identification", "fold": 8, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_1},
    {"name": "Mistake Location", "fold": 4, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_2},
    {"name": "Actionability", "fold": 1, "mode": "synth", "output_dir": OUTPUT_ROOT_TASK_4},
]

predictions = run_task_predictions(custom_history, custom_response, task_configs)

for task_name, payload in predictions.items():
    print(f"{task_name} -> Checkpoint: {payload['checkpoint']}")
    print(f"{task_name} -> Predicted label: {payload['predicted_label']}")
    print("Class probabilities:")
    for label, prob in payload["probabilities"].items():
        print(f"  {label:>15}: {prob:.4f}")
    print()


Mistake Identification -> Checkpoint: BD\mpnet_comparison_task1\without_synth_fold_8\checkpoint-1400
Mistake Identification -> Predicted label: Yes
Class probabilities:
               No: 0.0044
   To some extent: 0.0182
              Yes: 0.9774

Mistake Location -> Checkpoint: BD\mpnet_comparison_task2\without_synth_fold_4\checkpoint-798
Mistake Location -> Predicted label: Yes
Class probabilities:
               No: 0.1037
   To some extent: 0.0539
              Yes: 0.8424

Actionability -> Checkpoint: BD\mpnet_comparison_task4\synth_fold_1\checkpoint-2502
Actionability -> Predicted label: No
Class probabilities:
               No: 0.9953
   To some extent: 0.0020
              Yes: 0.0027



# Preload BD Model (Optimization)


In [9]:
class ModelManager:
    """Manages preloaded models for efficient evaluation"""
    
    def __init__(self):
        self.bd_models = {}
        self.ialabuc_model = None
        
    def load_bd_models(self, task_configs):
        """Preload all BD task models"""
        for config in task_configs:
            key = config["name"]
            self.bd_models[key] = load_latest_checkpoint(
                mode=config["mode"],
                fold=config["fold"],
                output_dir=config["output_dir"]
            )
        
    def load_all_models(self, task_configs):
        """Load all models at once"""
        self.load_bd_models(task_configs)
        
    def get_bd_model(self, task_name):
        """Get a preloaded BD model"""
        return self.bd_models.get(task_name)
    

# Initialize global model manager
model_manager = ModelManager()

# IALab UC (Correa Busquets et al., 2025) Configuration

Correa Busquets, S., Córdova Véliz, V., Baier, J.: IALab UC at BEA 2025 shared task: LLM-powered expert pedagogical feature extraction. In: Proceedings of the 20th Workshop on Innovative Use of NLP for Building Educational Applications (BEA 2025). pp. 1187–1193. ACL (2025)


In [6]:
import re

In [7]:
# === Helper functions ===
OPTIONS = ("0", "1")
question_pattern = r"(?i)^Question: .+\?\s"
label_pattern = r"(?i)^((The )?(Annotation )?label( is)?|Multiple choice)\W*"
full_sentence_answer_pattern = r"(?i)^The( (correct|best))? ?answer( here)?( is)?\W*"
much_ado_pattern = r"(?i)^.*the final answer is\W*"
explanation_pattern = r"(?i)^.*why the answer is\W*"
emphasis_pattern = r"(?i)^([*]+|strong>)"
answer_pattern = r"(?i)^answer\W*"
endgarbage_pattern = r"\W+$"
final0_pattern = r"\b0\b.{0,20}$"
final1_pattern = r"\b1\b.{0,20}$"
mid0_pattern = r"(?i)answer.{1,30}\b0\b.*$"
mid1_pattern = r"(?i)answer.{1,30}\b1\b.*$"
beginning0_pattern = r"(?i)response.{1,30}\b0\b.*$"
beginning1_pattern = r"(?i)response.{1,30}\b1\b.*$"

def strip_gen(text):
    '''Normalizes LLM generation to binary answer.'''
    stripped = re.sub(question_pattern, "", str(text))
    stripped = re.sub(label_pattern, "", stripped)
    stripped = re.sub(full_sentence_answer_pattern, "", stripped)
    stripped = re.sub(emphasis_pattern, "", stripped)
    stripped = re.sub(answer_pattern, "", stripped)
    stripped = re.sub(endgarbage_pattern, "", stripped)
    for opt in OPTIONS:
        if stripped.startswith(opt) or stripped.endswith(opt):
            return opt
    stripped = re.sub(much_ado_pattern, "", stripped)
    stripped = re.sub(explanation_pattern, "", stripped)
    stripped = re.sub(final0_pattern, "0", stripped)
    stripped = re.sub(final1_pattern, "1", stripped)
    for opt in OPTIONS:
        if stripped.startswith(opt) or stripped.endswith(opt):
            return opt
    stripped = re.sub(mid0_pattern, "0", stripped)
    stripped = re.sub(mid1_pattern, "1", stripped)
    for opt in OPTIONS:
        if stripped.endswith(opt):
            return opt
    stripped = re.sub(beginning0_pattern, "0", stripped)
    stripped = re.sub(beginning1_pattern, "1", stripped)
    for opt in OPTIONS:
        if stripped.endswith(opt):
            return opt
    return stripped

def convert_to_neg1_pos1(X):
    if X == "0" or X == 0:
        return -1.0
    elif X == "1":
        return 1.0
    return X

In [ ]:
# Configure Ollama client (runs on localhost by default)
client_qwen = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
model_id = "qwen3:4b-instruct"

# === Feature questions for LLM ===
feature_questions = {
    # Metacognition
    'tr_metacognition': 'Does the tutor\'s final response show preference for asking, rather than stating, to the student what their error could have been and/or how to fix it?',
    
    # Student's direction
    'loc_achieved_expresses': 'Does the tutor\'s final response express that the student has taken some steps correctly?',
    'loc_achieved_singlesout': 'Is the tutor\'s final response specific about which portion of the student\'s messages are going in the right direction to solve the proposed problem?',
    'loc_achieved_correct': 'Is the tutor\'s final response correct about which portion of the student\'s messages are going in the right direction to solve the proposed problem?',
    
    # Explanation
    'loc_mistaken_provides': 'Does the tutor\'s final response provide an explanation for why the student\'s approach was incorrect?',
    'loc_mistaken_provides_clear': 'Regarding the tutor\'s explanation for why the student\'s approach was incorrect, is it clear and understandable at a 6th grade level?',
    'loc_mistaken_provides_accurate': 'Regarding the tutor\'s explanation for why the student\'s approach was incorrect, is it fully accurate?',
    
    # Examples and strategies
    'loc_mistaken_offers': 'Does the tutor\'s final response offer the student a correct strategy or hint to solve the word problem?',
    'loc_mistaken_provides_example': 'Does the tutor\'s final response offer the student an example problem or fact to correct a misinterpretation of the original problem?',
    
    # Revealing answer
    'loc_revealing_answer': 'Does the tutor\'s final response reveal or includes the answer?',
}


def query_llm(question, conversation_history, tutor_response):
    '''Query Ollama LLM to classify a feature'''
    evaluation_prompt = f"""You will be presented with the conversation history from a grade-school math tutoring session happening over computer chat, where the student makes a mistake or evidences confusion.
Your task is to evaluate only the tutor's final response in terms of the question:
{question}
----------------------------------
Conversation history:
{conversation_history}
----------------------------------
Tutor's final response:
{tutor_response}
----------------------------------
Question: {question} (answer only with "0" for No, or "1" for Yes)
Answer: """
    
    try:
        completion = client_qwen.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "user",
                    "content": evaluation_prompt
                }
            ],
            temperature=0.1,
        )
        raw_answer = completion.choices[0].message.content.strip()
        normalized = strip_gen(raw_answer)
        return normalized
    except Exception as e:
        print(f"Error querying LLM: {e}")
        return "0"

In [ ]:
# === Feature Category Weights ===
# Define categories and their importance in determining "Providing Guidance"
# There is no correct weighting scheme
FEATURE_CATEGORIES = {
    "Metacognition": {
        "features": ["tr_metacognition"],
        "weight": 0.20,  # 20% importance
        "description": "Asking vs. stating to help student discover errors"
    },
    "Student's Direction": {
        "features": ["loc_achieved_expresses", "loc_achieved_singlesout", "loc_achieved_correct"],
        "weight": 0.20,  # 20% importance
        "description": "Recognizing and specifying correct steps taken"
    },
    "Explanation": {
        "features": ["loc_mistaken_provides", "loc_mistaken_provides_clear", "loc_mistaken_provides_accurate"],
        "weight": 0.20,  # 20% importance
        "description": "Explaining mistakes clearly and accurately"
    },
    "Examples and Strategies": {
        "features": ["loc_mistaken_offers", "loc_mistaken_provides_example"],
        "weight": 0.20,  # 20% importance
        "description": "Providing strategies, hints, or examples"
    },
    "Revealing Answer": {
        "features": ["loc_revealing_answer"],
        "weight": -0.20,  # -20% (- as it is inverse)
        "description": "Should NOT reveal the final answer"
    }
}


def calculate_category_scores(feature_values, categories=FEATURE_CATEGORIES):
    """
    Calculate composite score for each category based on individual feature values.
    
    Args:
        feature_values: dict of feature_name -> value (-1, 0, or 1)
        categories: Category definitions with features and weights
    
    Returns:
        dict: Category scores and overall score
    """
    category_scores = {}
    
    for category_name, category_info in categories.items():
        features = category_info["features"]
        feature_vals = [feature_values.get(f, 0) for f in features]
        
        valid_vals = [v for v in feature_vals if v != 0]
        if valid_vals:
            category_score = sum(valid_vals) / len(valid_vals)
        else:
            category_score = 0
        
        category_scores[category_name] = {
            "score": category_score,
            "feature_values": {f: feature_values.get(f, 0) for f in features}
        }
    
    return category_scores


def predict_label_from_features(feature_values, label_map=None):
    """
    Predict class probabilities for "Yes", "To some extent", or "No" based on weighted feature scores.
    
    Args:
        feature_values: dict of feature_name -> value (-1, 0, or 1)
        label_map: Optional custom label map (not used in probability output)
    
    Returns:
        dict: Contains class_probabilities, weighted_score, category_scores, and quality metrics
    """
    if label_map is None:
        label_map = {-1: "No", 0: "To some extent", 1: "Yes"}
    
    # Calculate category scores
    category_scores = calculate_category_scores(feature_values)
    
    # Calculate weighted overall score
    weighted_score = sum(
        category_scores[cat_name]["score"] * FEATURE_CATEGORIES[cat_name]["weight"]
        for cat_name in category_scores.keys()
    )
    
    # Normalize weights for interpretation
    total_weight = sum(FEATURE_CATEGORIES[cat_name]["weight"] for cat_name in category_scores.keys() if FEATURE_CATEGORIES[cat_name]["weight"] > 0)
    normalized_score = weighted_score / total_weight if total_weight > 0 else 0
    
    # Convert weighted score to class probabilities using softmax
    temperature = 0.5  # Controls sharpness of probability distribution
    
    # Compute logits for each class based on weighted score
    logit_yes = weighted_score / temperature
    logit_no = -weighted_score / temperature
    logit_between = -abs(weighted_score) / (2 * temperature)
    
    # Apply softmax for probability distribution
    logits = np.array([logit_yes, logit_between, logit_no])
    logits_shifted = logits - np.max(logits)
    exp_logits = np.exp(logits_shifted)
    probabilities = exp_logits / np.sum(exp_logits)
    
    class_probabilities = {
        "Yes": float(probabilities[0]),
        "To some extent": float(probabilities[1]),
        "No": float(probabilities[2])
    }
    
    max_label = max(class_probabilities.items(), key=lambda x: x[1])[0]
    
    # Check for hard constraints
    revealing_answer = feature_values.get("loc_revealing_answer", 0)
    # Handle hard constraint case - if revealing answer and would have been "Yes", downgrade to "To some extent"
    if revealing_answer == 1:
        # Determine which class had the highest probability
        if max_label == "Yes":
            # Redistribute "Yes" probability to "To some extent"
            yes_prob = class_probabilities["Yes"]
            class_probabilities["Yes"] = 0.05
            class_probabilities["To some extent"] += (yes_prob - 0.05)
            max_label = "To some extent"
        # else: keep the calculated probabilities unchanged
    
    return {
        "class_probabilities": class_probabilities,
        "weighted_score": float(weighted_score),
        "normalized_score": float(normalized_score),
        "category_scores": category_scores,
        "predicted_label": max_label,
    }

In [11]:
def predict_providing_guidance(custom_history, custom_response):
    """
    Predict Providing_Guidance class probabilities for a given tutor-student interaction.
    
    Args:
        custom_history: Conversation history string
        custom_response: Tutor's response string
    
    Returns:
        dict: Contains class_probabilities, feature_values, category_scores, and quality metrics
    """
    
    def process_single_feature(question, label, conversation_history, tutor_response):
        '''Process a single feature query'''
        raw_value = query_llm(question, conversation_history, tutor_response)
        normalized_value = convert_to_neg1_pos1(raw_value)
        return label, raw_value, normalized_value

    # Process all features in parallel
    custom_feature_values = {}
    max_workers = 4

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_feature = {
            executor.submit(process_single_feature, feature, label, custom_history, custom_response): label
            for label, feature in feature_questions.items()
        }
        
        for future in as_completed(future_to_feature):
            feature_name = future_to_feature[future]
            try:
                feature, raw_value, normalized_value = future.result()
                custom_feature_values[feature] = normalized_value
            except Exception as e:
                custom_feature_values[feature_name] = 0  # Default to uncertain on error

    # Predict label from features
    prediction = predict_label_from_features(custom_feature_values)
    
    return {
        "predicted_label": prediction["predicted_label"],
        "class_probabilities": prediction["class_probabilities"],
        "feature_values": custom_feature_values,
        "weighted_score": prediction["weighted_score"],
    }

### Custom example for prediction


In [ ]:
# === Example usage ===
custom_history = """
Tutor: Great! Plus 1 Point. \n Tutor: V = 2 x 2 x 13. \n Tutor: What is 2 x 2 x 13? \n Student: 49
"""

custom_response = """
Tutor: That's close! Remember, when we multiply 2 x 2, we get 4. Now lets try multiplying 4 by 13. What do you get when you multiply 4 by 13? The answer is 52
"""

result = predict_providing_guidance(custom_history, custom_response)

for key, value in result["feature_values"].items():
    print(f"{key}: {value}")

print("\n" + "="*50)
print("PREDICTION RESULTS")
print("="*50)
print(f"\nCustom History:\n{custom_history}\n")
print(f"Custom Response:\n{custom_response}\n")
print("Providing Guidance - Class Probabilities:")
for label, prob in result['class_probabilities'].items():
    print(f"  {label:>15}: {prob:.4f}")
print(f"\nWeighted Score: {result['weighted_score']:.4f}")
print("="*50)

loc_achieved_expresses: 1.0
loc_achieved_correct: 1.0
loc_achieved_singlesout: 1.0
tr_metacognition: 1.0
loc_mistaken_provides: 1.0
loc_mistaken_provides_clear: 1.0
loc_mistaken_provides_accurate: 1.0
loc_mistaken_offers: 1.0
loc_mistaken_provides_example: 1.0
loc_revealing_answer: 1.0

PREDICTION RESULTS

Custom History:

Tutor: Great! Plus 1 Point. 
 Tutor: V = 2 x 2 x 13. 
 Tutor: What is 2 x 2 x 13? 
 Student: 49


Custom Response:

Tutor: That's close! Remember, when we multiply 2 x 2, we get 4. Now lets try multiplying 4 by 13. What do you get when you multiply 4 by 13? The answer is 52


Providing Guidance - Class Probabilities:
              Yes: 0.0500
   To some extent: 0.8778
               No: 0.0722

Weighted Score: 0.6000


# Pedagogical Feedback Evaluation for the 4 tasks


In [12]:
def evaluate_conversation(custom_history, custom_response, task_configs=None, model_mgr=None):
    """
    Evaluate a tutor-student conversation using both BD tasks and IALab UC prediction.
    
    Args:
        custom_history: Conversation history string
        custom_response: Tutor's response string
        task_configs: Optional list of task configurations for BD predictions.
                     If None, uses default configs for all 3 tasks.
        model_mgr: Optional ModelManager with preloaded models for efficiency
    
    Returns:
        dict: Contains 'bd_predictions' and 'ialabuc_prediction'
    """
    # Default task configurations if none provided
    if task_configs is None:
        task_configs = [
            {"name": "Mistake Identification", "fold": 8, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_1},
            {"name": "Mistake Location", "fold": 4, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_2},
            {"name": "Actionability", "fold": 1, "mode": "synth", "output_dir": OUTPUT_ROOT_TASK_4},
        ]
    
    # Get preloaded models if available
    preloaded_bd_models = None
    
    if model_mgr is not None:
        preloaded_bd_models = model_mgr.bd_models
    
    # Run BD and IALab UC predictions in parallel
    with ThreadPoolExecutor(max_workers=2) as executor:
        bd_future = executor.submit(
            run_task_predictions, 
            custom_history, 
            custom_response, 
            task_configs,
            preloaded_bd_models
        )
        ialabuc_future = executor.submit(
            predict_providing_guidance, 
            custom_history, 
            custom_response
        )
        
        bd_predictions = bd_future.result()
        ialabuc_prediction = ialabuc_future.result()
    
    return {
        "bd_predictions": bd_predictions,
        "ialabuc_prediction": ialabuc_prediction
    }


def print_evaluation_results(custom_history, custom_response, results):
    """Print formatted evaluation results."""
    print("\n" + "="*70)
    print("COMPREHENSIVE EVALUATION RESULTS")
    print("="*70)
    
    print(f"\nCustom History:\n{custom_history}\n")
    print(f"Custom Response:\n{custom_response}\n")
    
    # Print BD predictions
    print("-" * 70)
    print("BD TASK PREDICTIONS")
    print("-" * 70)
    for task_name, payload in results["bd_predictions"].items():
        print(f"\n{task_name}:")
        print(f"  Predicted label: {payload['predicted_label']}")
        print("  Class probabilities:")
        for label, prob in payload["probabilities"].items():
            print(f"    {label:>15}: {prob:.4f}")
    
    # Print IALab UC prediction
    print("\n" + "-" * 70)
    print("IALABUC PROVIDING_GUIDANCE PREDICTION")
    print("-" * 70)
    ialabuc = results["ialabuc_prediction"]
    print(f"\nPredicted Providing_Guidance: {ialabuc['predicted_label']}")
    print("\nClass Probabilities:")
    for label, prob in ialabuc['class_probabilities'].items():
        print(f"  {label:>15}: {prob:.4f}")
    print("\n" + "="*70)


### Custom example for prediction on the 4 tasks


In [ ]:
custom_history = """
Tutor: You earn one point for your good beginning. \n Tutor: That was a good try. \n Tutor: What is the value of 3^3? \n Student: 9"
"""

custom_response = """
You are wrong!
"""

# Run comprehensive evaluation
results = evaluate_conversation(custom_history, custom_response)

# Print results
print_evaluation_results(custom_history, custom_response, results)


COMPREHENSIVE EVALUATION RESULTS

Custom History:

Tutor: You earn one point for your good beginning. 
 Tutor: That was a good try. 
 Tutor: What is the value of 3^3? 
 Student: 9"


Custom Response:

You are wrong!


----------------------------------------------------------------------
BD TASK PREDICTIONS
----------------------------------------------------------------------

Mistake Identification:
  Predicted label: Yes
  Class probabilities:
                 No: 0.0047
     To some extent: 0.0188
                Yes: 0.9765

Mistake Location:
  Predicted label: No
  Class probabilities:
                 No: 0.9379
     To some extent: 0.0333
                Yes: 0.0288

Actionability:
  Predicted label: Yes
  Class probabilities:
                 No: 0.0048
     To some extent: 0.0056
                Yes: 0.9896

----------------------------------------------------------------------
IALABUC PROVIDING_GUIDANCE PREDICTION
------------------------------------------------------------

In [53]:
results

{'bd_predictions': {'Mistake Identification': {'checkpoint': 'BD\\mpnet_comparison_task1\\without_synth_fold_8\\checkpoint-1400',
   'predicted_label': 'Yes',
   'probabilities': {'No': 0.0047372449189424515,
    'To some extent': 0.018763015046715736,
    'Yes': 0.9764997959136963}},
  'Mistake Location': {'checkpoint': 'BD\\mpnet_comparison_task2\\without_synth_fold_4\\checkpoint-798',
   'predicted_label': 'No',
   'probabilities': {'No': 0.9378737211227417,
    'To some extent': 0.03334202244877815,
    'Yes': 0.02878427505493164}},
  'Actionability': {'checkpoint': 'BD\\mpnet_comparison_task4\\synth_fold_1\\checkpoint-2502',
   'predicted_label': 'Yes',
   'probabilities': {'No': 0.004770076368004084,
    'To some extent': 0.005628926679491997,
    'Yes': 0.9896009564399719}}},
 'ialabuc_prediction': {'predicted_label': 'No',
  'class_probabilities': {'Yes': 0.07222670133967153,
   'To some extent': 0.13160563040120588,
   'No': 0.7961676682591227},
  'feature_values': {'loc_achie

# Rubric-Guided Feedback Refinement


In [ ]:
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)

def refine_tutor_response(history, response, task_configs=None, max_iterations=3, model_mgr=None):
    """
    Refine a tutor response iteratively using all 4 evaluation dimensions.
    
    Args:
        history: Conversation history string
        response: Initial tutor response string
        task_configs: Optional task configurations for BD predictions
        max_iterations: Maximum number of refinement iterations
        model_mgr: Optional ModelManager with preloaded models for efficiency
    
    Returns:
        dict: Contains predictions, refined_response, iterations, and completion status
    """
    custom_history = history.strip()
    current_response = response.strip()

    refinement_labels = {"No", "To some extent"}
    iteration_records = []

    def request_refinement(all_predictions):
        evaluation_summary = []
        
        # Add BD predictions
        for task_name, payload in all_predictions["bd_predictions"].items():
            probs_formatted = ", ".join(
                f"{label}={prob:.2%}" for label, prob in payload["probabilities"].items()
            )
            evaluation_summary.append(
                f"{task_name}: {payload['predicted_label']} ({probs_formatted})"
            )
        
        # Add IALab UC prediction
        ialabuc = all_predictions["ialabuc_prediction"]
        probs_formatted = ", ".join(
            f"{label}={prob:.2%}" for label, prob in ialabuc["class_probabilities"].items()
        )
        evaluation_summary.append(
            f"Providing Guidance (should provide the student with relevant and helpful guidance): {ialabuc['predicted_label']} ({probs_formatted})"
        )
        
        summary_block = "\n".join(evaluation_summary)

        prompt = f"""The following tutor reply needs refinement because at least one evaluator flagged it.

Conversation history:
{custom_history}

Original tutor response:
{current_response}

Evaluation summary:
{summary_block}

Respond ONLY with the refined and concise tutor response directly addressing the student's misconception, pinpoint its location, give next steps, provide guidance without revealing the answer, and remain encouraging, but not overpraising.
"""

        response_payload = client.chat.completions.create(
            model=OLLAMA_MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are a math tutor who improves feedback for struggling students.",
                },
                {"role": "user", "content": prompt.strip()},
            ],
            temperature=0.3,
        )
        reply = response_payload.choices[0].message.content if response_payload.choices else ""
        refined = reply.strip() or "<empty response>"
        return refined

    all_predictions = None
    for iteration in range(1, max_iterations + 1):
        
        all_predictions = evaluate_conversation(custom_history, current_response, task_configs, model_mgr)
        
        iteration_records.append(
            {
                "iteration": iteration,
                "response": current_response,
                "predictions": all_predictions,
            }
        )
        
        ialabuc = all_predictions["ialabuc_prediction"]

        # Check if any dimension needs refinement
        bd_needs_refinement = any(
            payload["predicted_label"] in refinement_labels 
            for payload in all_predictions["bd_predictions"].values()
        )
        ialabuc_needs_refinement = ialabuc["predicted_label"] in refinement_labels
        needs_refinement = bd_needs_refinement or ialabuc_needs_refinement

        if not needs_refinement:
            return {
                "predictions": all_predictions,
                "refined_response": current_response,
                "iterations": iteration_records,
                "completed": True,
            }

        if iteration == max_iterations:
            break

        try:
            current_response = request_refinement(all_predictions)
        except Exception as exc:
            print(f"Refinement request failed: {exc}")
            return {
                "predictions": all_predictions,
                "refined_response": current_response,
                "iterations": iteration_records,
                "completed": False,
                "error": str(exc),
            }

    return {
        "predictions": all_predictions,
        "refined_response": current_response,
        "iterations": iteration_records,
        "completed": False,
    }


### Custom example for prediction and refinement


In [ ]:
# === Preload all models for optimal performance ===
task_configs = [
    {"name": "Mistake Identification (includes the relevant mistake identification)", "fold": 8, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_1},
    {"name": "Mistake Location (points to the mistake's location in the answer and outline what the error is to help the student remediate)", "fold": 2, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_2},
    {"name": "Actionability (provides what the student should do next)", "fold": 1, "mode": "synth", "output_dir": OUTPUT_ROOT_TASK_4},
]

model_manager.load_all_models(task_configs)

# === Test refinement with preloaded models ===
custom_history = """
Tutor: Great! Plus 1 Point. \n Tutor: V = 2 x 2 x 13. \n Tutor: What is 2 x 2 x 13? \n Student: 49
"""
custom_response = """
Your are wrong! It is 52.
"""

result = refine_tutor_response(
    custom_history, 
    custom_response, 
    task_configs, 
    max_iterations=3,
    model_mgr=model_manager  # Pass preloaded models
)
print(f"\nCompleted: {result.get('completed')}")
print('\nFinal Tutor response:')
print(result.get("refined_response"))



Completed: True

Final Tutor response:
It looks like the multiplication step went wrong.  
First multiply **2 × 2**.  
Then take that result and multiply it by **13**.  

Give it another try and see where the numbers differ. You’re on the right track—just double‑check each step.


In [68]:
result

{'predictions': {'bd_predictions': {'Mistake Identification (includes the relevant mistake identification)': {'predicted_label': 'Yes',
    'probabilities': {'No': 0.004428476560860872,
     'To some extent': 0.016966737806797028,
     'Yes': 0.978604793548584}},
   "Mistake Location (points to the mistake's location in the answer and outline what the error is to help the student remediate)": {'predicted_label': 'Yes',
    'probabilities': {'No': 0.05562631040811539,
     'To some extent': 0.08695831894874573,
     'Yes': 0.8574153780937195}},
   'Actionability (provides what the student should do next)': {'predicted_label': 'Yes',
    'probabilities': {'No': 0.0028741408605128527,
     'To some extent': 0.006570336874574423,
     'Yes': 0.9905555844306946}}},
  'ialabuc_prediction': {'predicted_label': 'Yes',
   'class_probabilities': {'Yes': 0.8837980883506893,
    'To some extent': 0.08017635369626987,
    'No': 0.03602555795304092},
   'feature_values': {'loc_achieved_expresses': 1

# Evaluation and Refinement on a Dataset


In [27]:
from tqdm import tqdm

dataset_path = Path("BEA Dataset") / "mrbench_v3_devset.json"
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "mrbench_v3_devset_refinements.jsonl"
checkpoint_path = output_dir / "checkpoint.json"

task_configs = [
    {"name": "Mistake Identification (includes the relevant mistake identification)", "fold": 8, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_1},
    {"name": "Mistake Location (points to the mistake's location in the answer and outline what the error is to help the student remediate)", "fold": 4, "mode": "without_synth", "output_dir": OUTPUT_ROOT_TASK_2},
    {"name": "Actionability (provides what the student should do next)", "fold": 1, "mode": "synth", "output_dir": OUTPUT_ROOT_TASK_4},
]

def _extract_entries(blob):
    if isinstance(blob, list):
        return blob
    if isinstance(blob, dict):
        if isinstance(blob.get("data"), list):
            return blob["data"]
        return [blob]
    return []

def load_checkpoint(checkpoint_file):
    """Load the last processed conversation_id from checkpoint"""
    if checkpoint_file.exists():
        with checkpoint_file.open("r", encoding="utf-8") as f:
            data = json.load(f)
            return set(data.get("processed_ids", []))
    return set()

def save_checkpoint(checkpoint_file, processed_ids):
    """Save the list of processed conversation_ids"""
    with checkpoint_file.open("w", encoding="utf-8") as f:
        json.dump({"processed_ids": list(processed_ids)}, f, ensure_ascii=False, indent=2)

def create_bea_refinements(dataset_file, destination_file, checkpoint_file, task_configs, max_iterations=4, model_mgr=None):
    """
    Process BEA dataset and generate refinements using preloaded models.
    Supports resuming from checkpoint if interrupted.
    
    Args:
        dataset_file: Path to input JSON dataset
        destination_file: Path to output JSONL file
        checkpoint_file: Path to checkpoint file for resume capability
        task_configs: Task configurations for BD predictions
        max_iterations: Maximum refinement iterations per response
        model_mgr: ModelManager with preloaded models (significantly faster)
    """
    dataset_file = Path(dataset_file)
    destination_file = Path(destination_file)
    checkpoint_file = Path(checkpoint_file)
    
    if not dataset_file.exists():
        raise FileNotFoundError(f"Missing dataset file: {dataset_file}")

    # Load checkpoint to resume if needed
    processed_ids = load_checkpoint(checkpoint_file)
    if processed_ids:
        print(f"Found checkpoint with {len(processed_ids)} already processed conversations. Resuming...")
        file_mode = "a"  # Append mode
    else:
        print("Starting fresh processing...")
        file_mode = "w"  # Write mode

    with dataset_file.open("r", encoding="utf-8") as source_fp:
        raw_blob = json.load(source_fp)

    entries = _extract_entries(raw_blob)
    if not entries:
        raise ValueError("Dataset does not contain any entries to process.")

    processed = 0
    
    with destination_file.open(file_mode, encoding="utf-8") as sink_fp:
        # Use tqdm for progress tracking
        for sample in tqdm(entries, desc="Processing conversations", unit="conversation"):
            conversation_id = sample.get("conversation_id")
            
            # Skip if already processed
            if conversation_id in processed_ids:
                continue
            
            history = sample.get("conversation_history", "").strip()
            annotated_replies = sample.get("tutor_responses") or {}
            if not history or not annotated_replies:
                continue

            # Add tqdm for model responses within each conversation
            model_items = list(annotated_replies.items())
            for model_alias, payload in model_items:
                tutor_reply = (payload or {}).get("response", "").strip()
                if not tutor_reply:
                    continue

                try:
                    refinement = refine_tutor_response(
                        history,
                        tutor_reply,
                        task_configs=task_configs,
                        max_iterations=max_iterations,
                        model_mgr=model_mgr  # Use preloaded models for efficiency
                    )

                    sink_fp.write(
                        json.dumps(
                            {
                                "conversation_id": conversation_id,
                                "model_alias": model_alias,
                                "original_response": tutor_reply,
                                "refined_response": refinement.get("refined_response"),
                                "completed": refinement.get("completed"),
                                "iterations": refinement.get("iterations"),
                                "predictions": refinement.get("predictions"),
                            },
                            ensure_ascii=False,
                        )
                        + "\n"
                    )
                    sink_fp.flush()  # Ensure data is written to disk
                    processed += 1
                    
                except Exception as e:
                    tqdm.write(f"Error processing conversation {conversation_id} (model: {model_alias}): {e}")
                    continue
            
            # Mark conversation as processed and save checkpoint
            processed_ids.add(conversation_id)
            save_checkpoint(checkpoint_file, processed_ids)

    return processed


In [28]:
# === Preload models once for the entire dataset processing ===
bea_model_manager = ModelManager()
bea_model_manager.load_all_models(task_configs)

# === Process dataset with preloaded models ===
total_refinements = create_bea_refinements(
    dataset_file=dataset_path,
    destination_file=output_path,
    checkpoint_file=checkpoint_path,
    task_configs=task_configs,
    max_iterations=3,
    model_mgr=bea_model_manager  # Pass preloaded models
)
print(f"\n{'='*70}")
print(f"Generated {total_refinements} refined responses -> {output_path}")
print(f"Checkpoint saved to -> {checkpoint_path}")
print(f"{'='*70}")


Found checkpoint with 295 already processed conversations. Resuming...


Processing conversations: 100%|██████████| 300/300 [24:43<00:00,  4.95s/conversation]


Generated 41 refined responses -> outputs\mrbench_v3_devset_refinements.jsonl
Checkpoint saved to -> outputs\checkpoint.json
